# data-collection_006

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (67).ipynb`

| Field | Value |
|---|---|
| Section | `data_collection` |
| Market | `us_equities` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 15 |
| Detected functions | — |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas_datareader as pdr
import yfinance as yf
import datetime

In [ ]:
df = pd.read_csv('ICICIBANK_minute_data_with_indicators.csv')
df

In [ ]:
# Ensure 'close' and 'SMA' columns are numeric
df['close'] = pd.to_numeric(df['close'], errors='coerce')
df['SMA'] = df['close'].rolling(3).mean()

# Calculate returns and ratios
df['Simple_Returns'] = df['close'].pct_change(1)
df['log_returns'] = np.log(1 + df['Simple_Returns'])
df['Ratios'] = df['close'] / df['SMA']

# Display the DataFrame
df


In [ ]:
df['Ratios'].describe()

In [ ]:
plt.plot(df.close)

In [ ]:
plt.figure(figsize=(14, 7))
df['Ratios'].dropna().plot(legend = True)

In [ ]:
# Set buy and sell signals based on the ratio
df['Positions'] = np.where(df['Ratios'] > 1.005, -1, np.nan)
df['Positions'] = np.where(df['Ratios'] < 0.995, 1, df['Positions'])
df['Positions'] = np.where(df['Ratios'] == 1, 0, df['Positions'])  # Close all positions when ratio = 1

# Forward-fill the positions to maintain the last known position
df['Positions'] = df['Positions'].ffill()

# Mark buy and sell prices based on the position
df['buy'] = np.where(df['Positions'] == 1, df['close'], np.nan)
df['sell'] = np.where(df['Positions'] == -1, df['close'], np.nan)
df['close'] = np.where(df['Positions'] == 0, df['close'], np.nan)

# Display the updated DataFrame
df


In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(df['close'], alpha = 0.5, label = "close")
plt.plot(df['SMA'], alpha=0.5, label = "SMA")
plt.scatter(df.index, df['buy'], color = 'g', marker = '^', alpha = 1)
plt.scatter(df.index, df['sell'], color = 'r', marker = 'v', alpha = 1)
plt.legend()
plt.show()

In [ ]:
df['Statergy_Returns'] = df.Positions.shift(1) * df.log_returns
df['Statergy_Returns']

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(np.exp(df['log_returns'].dropna()).cumprod(), color = 'green', label = 'Buy/Hold Statergy')
plt.plot(np.exp(df['Simple_Returns'].dropna()).cumprod(), color = 'red', label = 'Mean Reversion')
plt.legend()
plt.show()

In [ ]:
# Buy and Hold Strategy Returns
buy_hold_returns = np.exp(df['log_returns'].dropna()).cumprod().iloc[-1] - 1
print('Buy/Hold Strategy Returns:', buy_hold_returns)

# Placeholder Mean Reversion Strategy Returns (note: this is not an actual mean reversion calculation)
mean_reversion_returns = np.exp(df['Simple_Returns'].dropna()).cumprod().iloc[-1] - 1
print('Mean Reversion Strategy:', mean_reversion_returns)


In [ ]:
df.buy.count()

In [ ]:
df.sell.count()